# Questão 6 - Demand forecast

In [88]:
#Premissas obrigatórias:

#O período de treino deve incluir dados até 31/12/2025.
#O período de teste deve ser o primeiro trimestre de 2026.
#A previsão deve ser feita em base mensal.
#Considere apenas o produto: "Bússola de Bordo 702".

In [89]:
#Importando as bibliotecas necessárias
import pandas as pd
from sklearn.metrics import mean_absolute_error


In [90]:
#1. Utilize os datasets products.csv, product_variants.csv, orders.csv e order_items.csv para criar um dataset unificado que facilite a criação do modelo preditivo.

#Importando datasets
products = pd.read_csv("Dataset/products.csv")
product_variants = pd.read_csv("Dataset/product_variants.csv")
orders = pd.read_csv("Dataset/orders.csv")
order_items = pd.read_csv("Dataset/order_items.csv")

#Combinado os datasets por PK -> FK
df_combined = (
    products[["id", "name"]]
    .merge(
        product_variants[["id", "product_id"]],
        left_on="id",
        right_on="product_id"
    )
    .merge(
        order_items[["order_id", "product_variant_id", "quantity"]],
        left_on="id_y",
        right_on="product_variant_id"
    )
    .merge(
        orders[["id", "placed_at"]],
        left_on="order_id",
        right_on="id"
    )
)

#Filtrando apenas a Bússola de Bordo 702
df_filt = df_combined[df_combined["name"] == "Bússola de Bordo 702"]
display(df_filt)

,id_x,name,id_y,product_id,order_id,product_variant_id,quantity,id,placed_at
21302,74,Bússola de Bordo 702,147,74,262,147,5,262,2022-04-01 07:44:41
21303,74,Bússola de Bordo 702,147,74,328,147,1,328,2023-03-07 03:00:18
21304,74,Bússola de Bordo 702,147,74,533,147,3,533,2024-06-20 11:12:42
21305,74,Bússola de Bordo 702,147,74,939,147,4,939,2023-04-27 03:56:27
21306,74,Bússola de Bordo 702,147,74,1380,147,8,1380,2025-03-04 18:05:26
...,...,...,...,...,...,...,...,...,...
70733,240,Bússola de Bordo 702,486,240,48402,486,5,48402,2023-09-11 08:55:31
70734,240,Bússola de Bordo 702,486,240,49247,486,3,49247,2026-01-01 17:25:33
70735,240,Bússola de Bordo 702,486,240,49308,486,5,49308,2020-12-17 18:07:48
70736,240,Bússola de Bordo 702,486,240,49338,486,10,49338,2025-06-07 06:27:52


In [91]:
#2. Construa um modelo baseline simples, utilizando: Média móvel dos últimos 3 meses de vendas (considerando apenas dados anteriores à data prevista).

#Consolidando vendas por mês
df_filt["placed_at"] = pd.to_datetime(df_filt["placed_at"])
df_filt["month"] = df_filt["placed_at"].dt.to_period("M")
vendas_mensais = (
    df_filt.groupby("month")["quantity"]
      .sum()
      .reset_index()
      .sort_values("month")
)

#Média movel dos ultimos 3 meses utilizando o shift para considerar apenas dados anteriores a data prevista
vendas_mensais["previsao"] = (
    vendas_mensais["quantity"]
    .shift(1)
    .rolling(window=3)
    .mean()
)
#Atenção: naturalmente os meses de 2020-01, 2020-02 e 2020-03 não terão previsão, pois não há dados anteriores a eles.

#Iniciando modelo de previsão com base na média móvel dos últimos 3 meses de vendas.
#O período de treino deve incluir dados até 31/12/2025.
treino = vendas_mensais[
    vendas_mensais["month"] <= "2025-12"
]["quantity"].tolist()

#O período de teste deve ser o primeiro trimestre de 2026.
teste = vendas_mensais[
    (vendas_mensais["month"] >= "2026-01") &
    (vendas_mensais["month"] <= "2026-03")
]["month"].tolist()

#Para aferir com a previsão (não considerar como resposta)
teste_comparativo = vendas_mensais[
    (vendas_mensais["month"] >= "2025-10") &
    (vendas_mensais["month"] <= "2026-03")
]
display(teste_comparativo)
#É necessário verificar até 2025-10, pois corresponde os 3 meses previstos antes de 2026-01.

,month,quantity,previsao
68,2025-10,34,24.333333
69,2025-11,60,29.333333
70,2025-12,22,41.666667
71,2026-01,79,38.666667
72,2026-02,68,53.666667
73,2026-03,60,56.333333


In [92]:
#3. Gere a previsão mensal de vendas para o primeiro trimestre de 2026.

#Lista para armazenar as previsões do treino
previsoes = []

#Previsão dos 3 meses do primeiro trimestre de 2026
for mes in teste:

    #Média dos últimos 3 meses disponíveis
    previsao = sum(treino[-3:]) / 3

    #Armazena a previsão
    previsoes.append({
        "month": mes,
        "previsao_modelo": previsao
    })

    #Adiciona a previsão ao histórico
    treino.append(previsao)

previsoes_2026 = pd.DataFrame(previsoes)
print(previsoes_2026)

     month  previsao_modelo
0  2026-01        38.666667
1  2026-02        40.222222
2  2026-03        33.629630


In [ ]:
#4. Compare as previsões com os valores reais do período de teste utilizando a métrica: MAE (Mean Absolute Error)

#Utilizando MAE (Mean Absolute Error) da biblioteca scikit-learn para comparar as previsões com os valores reais do período de teste.
teste_2 = vendas_mensais[
    (vendas_mensais["month"] >= "2026-01") &
    (vendas_mensais["month"] <= "2026-03")
]

mae = mean_absolute_error(
    teste_2["quantity"],
    previsoes_2026["previsao_modelo"]
)

print(f"MAE: {mae:.2f}")
#Atenção: em média, o modelo errou aproximadamente 31,49 unidades por mês. Isso pode ser ilustrado ao analisar o df teste_comparativo para 2026-01.
#Modelo subestimanda a demanda: para a Bussola analisada, 3 meses não parece adequado para um modelo baseline simples. 
#Talvez seja interessante analisar a média móvel de 6 meses, ou até mesmo 12 meses, para verificar se o modelo melhora.

MAE: 31.49


In [ ]:
#5. Responda objetivamente:
# a. O baseline é adequado para esse produto?
# Não. O baseline não é adequado para esse produto, pois apresentou previsões significativamente abaixo das vendas reais, com MAE de aproximadamente 31,49 unidades.

# b. Cite uma limitação desse método.
# Uma limitação é que a média móvel considera apenas os últimos 3 meses e não captura tendências ou sazonalidades, podendo subestimar a demanda quando há crescimento nas vendas.

In [ ]:
#Como o baseline foi construído?
#Foi utilizada uma média móvel dos últimos 3 meses de vendas para prever a demanda do mês seguinte. Para as previsões de 2026, o modelo foi aplicado de forma recursiva, utilizando as previsões anteriores quando os valores reais ainda não estavam disponíveis.

#Como evitou data leakage?
#As previsões foram calculadas utilizando somente dados anteriores ao mês previsto. Para fevereiro e março de 2026, foram utilizadas as previsões dos meses anteriores, e não seus valores reais.

#Uma limitação do modelo proposto:
#A média móvel considera apenas os últimos 3 meses e não captura tendências ou sazonalidade, podendo gerar previsões imprecisas quando há mudanças no comportamento das vendas.